# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [1]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [2]:
!omnivoice-demo --share

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-05-05 15:29:25,108 root INFO: Loading model from k2-fsa/OmniVoice, device=cuda ...
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-05 15:29:25,411 huggingface_hub.utils._http WARNING: Warning: You are se

## 3. Option B — Python API

### 3.1 Load Model

In [3]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [13]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving test.wav to test.wav
Uploaded: test.wav


In [6]:
audio = model.generate(
    text="kahani shuruat hoti hee ek villa se jisme ham dekhte hee do bando ko ek nam hee eli and dusre kaa naam hee rei and yee kuch dhundh rahe hee  ",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.5 Custom Gradio App for Voice Cloning

In [14]:
import gradio as gr
import soundfile as sf
import numpy as np # Added for creating dummy audio
import os # Import the os module

# Create a dummy test.wav file for demonstration
# You can replace this with your own 'test.wav' file in the Colab environment
# if you want a specific voice.
if not os.path.exists('test.wav'):
    dummy_audio = np.random.randn(24000).astype(np.float32) # 1 second of random noise
    sf.write('test.wav', dummy_audio, 24000)
    print("Created dummy 'test.wav' file.")

def clone_voice_gradio(text):
    # Using a hardcoded reference audio path as requested
    ref_audio_path = 'test.wav'

    print(f"Using hardcoded ref_audio_path: {ref_audio_path}") # Added for debugging
    if not os.path.exists(ref_audio_path):
        return f"Error: Reference audio file '{ref_audio_path}' not found."

    audio_output = model.generate(
        text=text,
        ref_audio=ref_audio_path,
    )

    # OmniVoice.generate returns a list with a numpy array. We need the numpy array.
    generated_audio_array = audio_output[0]
    sample_rate = 24000 # OmniVoice output sample rate

    return (sample_rate, generated_audio_array)


# Create the Gradio interface
iface = gr.Interface(
    fn=clone_voice_gradio,
    inputs=[
        gr.Textbox(label="Text to speak")
    ],
    outputs=gr.Audio(label="Generated Speech"),
    title="OmniVoice - Custom Voice Cloning (Hardcoded Reference)",
    description="Enter text to generate speech using a hardcoded reference voice from 'test.wav'."
)

# Launch the interface
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6e1a59d44c9c6f53d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
